# 04_12 Recommendation - ALS
Train and evaluate ALS implicit feedback.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train ALS va xem ket qua goi y top-N ro rang tren notebook.
- Muc tieu ky thuat: Hien thi metric train/val/test va bang recommendation mau.

In [1]:
import sys, asyncio
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
from IPython.display import display
spark=(SparkSession.builder.appName('04_12_als').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'recommendation'/'als'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'als_train')).select('user_idx','item_idx','rating').dropna()
val_df=spark.read.parquet(str(FEATURE_DIR/'als_val')).select('user_idx','item_idx','rating').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'als_test')).select('user_idx','item_idx','rating').dropna()
als=ALS(userCol='user_idx',itemCol='item_idx',ratingCol='rating',implicitPrefs=True,alpha=40.0,rank=64,regParam=0.05,maxIter=15,coldStartStrategy='nan',seed=42)
m=als.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
evaluator=RegressionEvaluator(labelCol='rating',predictionCol='prediction',metricName='rmse')
# Filter NaN predictions (from coldStartStrategy='nan') instead of dropping all rows
from pyspark.sql import functions as F
pred_val_eval=pred_val.where(F.col("prediction").isNotNull() & ~F.isnan(F.col("prediction")) & F.col("rating").isNotNull())
pred_test_eval=pred_test.where(F.col("prediction").isNotNull() & ~F.isnan(F.col("prediction")) & F.col("rating").isNotNull())
val_eval_has_rows=pred_val_eval.limit(1).count()>0
test_eval_has_rows=pred_test_eval.limit(1).count()>0
val_rmse=float(evaluator.evaluate(pred_val_eval)) if val_eval_has_rows else None
test_rmse=float(evaluator.evaluate(pred_test_eval)) if test_eval_has_rows else None
rmse_for_report=test_rmse if test_rmse is not None else val_rmse
metrics={'model_family':'recommendation','model_name':'ALS','val_rmse':val_rmse,'rmse':rmse_for_report,'test_rmse':test_rmse,'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count(),'val_eval_ready':bool(val_eval_has_rows),'test_eval_ready':bool(test_eval_has_rows),'val_eval_rows':pred_val_eval.count(),'test_eval_rows':pred_test_eval.count()}
print(metrics)
display(pd.DataFrame([metrics]))
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'recommendation_als.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
try:
    user_subset=train_df.select('user_idx').dropna().distinct().limit(30)
    rec_df=m.recommendForUserSubset(user_subset,10)
    rec_df.show(5, truncate=False)
    rec_rows=rec_df.limit(30).collect()
    rec_pdf=pd.DataFrame([r.asDict(recursive=True) for r in rec_rows])
    display(rec_pdf)
except Exception as e:
    print(f'Warning: skip recommendation preview due to: {e}')
spark.stop()

26/03/31 22:26:59 WARN Utils: Your hostname, Genius-Macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.2.18 instead (on interface en0)
26/03/31 22:26:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 22:26:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/03/31 22:27:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/31 22:27:00 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


26/03/31 22:27:09 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/31 22:27:09 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


26/03/31 22:27:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


{'model_family': 'recommendation', 'model_name': 'ALS', 'val_rmse': 1.1935348010082374, 'rmse': 1.1838343421215172, 'test_rmse': 1.1838343421215172, 'train_rows': 70003, 'val_rows': 14781, 'test_rows': 15001, 'val_eval_ready': True, 'test_eval_ready': True, 'val_eval_rows': 908, 'test_eval_rows': 921}


,model_family,model_name,val_rmse,rmse,test_rmse,train_rows,val_rows,test_rows,val_eval_ready,test_eval_ready,val_eval_rows,test_eval_rows
0,recommendation,ALS,1.193535,1.183834,1.183834,70003,14781,15001,True,True,908,921


+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_idx|recommendations                                                                                                                                                                                                  |
+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|38460   |[{10, 1.3318651E-9}, {27, 1.0328046E-9}, {35, 9.553757E-10}, {311, 8.4883933E-10}, {26, 8.150503E-10}, {513, 6.940819E-10}, {365, 6.9044886E-10}, {28, 6.4644956E-10}, {208, 6.4336864E-10}, {89, 6.0860333E-10}]|
|161     |[{1883, 0.50322616}, {36, 0.36896124}, {234, 0.30279183}, {90, 0.28340167}, {385, 0.27900836}, {218, 0.273

,user_idx,recommendations
0,38460,"[{'item_idx': 10, 'rating': 1.3318650626814588..."
1,161,"[{'item_idx': 1883, 'rating': 0.50322616100311..."
2,1911,"[{'item_idx': 121, 'rating': 0.860272824764251..."
3,35481,"[{'item_idx': 1014, 'rating': 0.45159283280372..."
4,43171,"[{'item_idx': 3, 'rating': 1.1420819845398e-09..."
5,49331,"[{'item_idx': 11, 'rating': 1.2167714613653402..."
6,51551,"[{'item_idx': 6, 'rating': 0.35162216424942017..."
7,3842,"[{'item_idx': 350, 'rating': 0.784095585346221..."
8,3962,"[{'item_idx': 7, 'rating': 7.476967378572397e-..."
9,15992,"[{'item_idx': 11, 'rating': 1.0394114458023296..."
